# Evalvaluate Predictions

In [ ]:
import os
import polars as pl
import numpy as np
from deeptan.stat.metrics import MetricsDictMaker

## Computing metrics

In [ ]:
path_home_deeptan = "/mnt/hdd2/homext/wuch/xn2p"
path_predictions = os.path.join(path_home_deeptan, "run", "predict")
path_raw_df = os.path.join(path_home_deeptan, "data", "raw_df")

### scMultiome

In [ ]:
dataset = "sc_multiome_minmi0.35_top2000"
path_true = os.path.join(path_raw_df, "scMultiome", "Ath_scMultiome_WT_split_strata")
path_orig_h5ad = os.path.join(path_true, "origin.h5ad")

metricsmaker_scmul = MetricsDictMaker(os.path.join(path_predictions, dataset), path_true, True, path_orig_h5ad)
metricsmaker_scmul.run()

## Plot

In [ ]:
from deeptan.stat.plots import kde_grid_plot, kde_grid_plot_data, metrics_plot, metrics_plot_data

%config InlineBackend.figure_format = 'retina'
%config InlineBackend.figure_dpi = 300

### Plot recon distribution

In [ ]:
_dim = "sample_metrics"
# dim = "feature_metrics"

_metrics = ["mse", "mae", "jsd", "pcc"]
_metrics_text = ["MSE", "MAE", "JSD", "PCC"]

x_lab = "multitask"
x_lab_text = "Multitask"

# y_labs = ["multitask_noguide", "focus_recon", "focus_label"]
# y_labs_text = ["Multitask (no SGG)", "Focus on reconstruction", "Focus on labelling"]
y_labs = ["focus_recon", "focus_label"]
y_labs_text = ["Focus on reconstruction", "Focus on labelling"]

_tasks = [x_lab] + y_labs

##### Choose a dataset to plot

In [ ]:
_tmp_dataset = metricsmaker_scmul
_split = "tst"
fig_name = f"fig.c2.scmul.{_split}.recon.dist"

data4kde = kde_grid_plot_data(_tmp_dataset, 42, _metrics, _tasks, _dim)
_fig = kde_grid_plot(data4kde, x_lab, y_labs, _metrics, _metrics_text, x_lab_text, y_labs_text, _dim, fig_name, ".fig/", _split)

### Plot all metrics

In [ ]:
_tmp_dataset = metricsmaker_scmul
_split = "tst"
fig_name = f"fig.c2.scmul.{_split}.metrics"

# _df_plot_allmetrics = metrics_plot_data(_tmp_dataset, 42)
_df_plot_allmetrics = metrics_plot_data(_tmp_dataset, _split)
_df_plot_allmetrics = _df_plot_allmetrics.filter(pl.col("Task") != "Multitask (no SGG)").filter(pl.col("Metric") != "Spearman")
metrics_plot(_df_plot_allmetrics, fig_name, ".fig/")

### Plot PaCMAP

In [ ]:
_tmp_dataset = metricsmaker_scmul
_seed = 42
_split = "tst"
fig_name = f"fig.c2.scmul.{_split}.cell_emb"

In [ ]:
x_lab = "multitask"
x_lab_text = "Multitask"

# y_labs = ["multitask_noguide", "focus_recon", "focus_label"]
# y_labs_text = ["Multitask (no SGG)", "Focus on reconstruction", "Focus on labelling"]
y_labs = ["focus_recon", "focus_label"]
y_labs_text = ["Focus on reconstruction", "Focus on labelling"]

_tasks = [x_lab] + y_labs
_tasks_text = [x_lab_text] + y_labs_text

#### Get cell embeddings for each task

In [ ]:
cell_embs = {}
_fnames = []
for _task in _tasks:
    _fname = _tmp_dataset.ident.filter((pl.col("task") == _task) & (pl.col("seed_num") == _seed) & (pl.col("split") == _split))["fname"].item()
    _fnames.append(_fname)
    cell_embs[_task] = _tmp_dataset.metrics_dict["prediction"][_fname]["g_embedding"]

#### Get predicted cell labels for each task

In [ ]:
# 获取所有文件中唯一的细胞类型标签
celltypes_uniq = _tmp_dataset.metrics_dict["prediction"][_fnames[0]]["label_names"][1:]
celltypes_uniq = [i.replace("ct_", "") for i in celltypes_uniq]
print(celltypes_uniq)

ys_pred_numeric = {}
ys_pred_text = {}
for _fname in _fnames:
    _task = _tmp_dataset.ident.filter(pl.col("fname")==_fname)["task"].item()
    ys_pred_numeric[_task] = _tmp_dataset.metrics_dict["prediction"][_fname]["y"].argmax(axis=1)
    ys_pred_text[_task] = [celltypes_uniq[i] for i in ys_pred_numeric[_task]]

#### Get true cell types

In [ ]:
y_true_text = _tmp_dataset.metrics_dict["true"][f"seed_{_seed}_tst"]["y_df_flatten"]["ct"].to_list()

#### Compute PaCMAP

In [ ]:
import pacmap

In [ ]:
embedding = pacmap.PaCMAP(n_components=2, n_neighbors=None, MN_ratio=0.3, FP_ratio=4.0)

# # fit the data (The index of transformed data corresponds to the index of the original data)
# X_transformed = embedding.fit_transform(X, init="pca")

cell_embs_pacmap = {}
for _task in cell_embs.keys():
    cell_embs_pacmap[_task] = embedding.fit_transform(cell_embs[_task], init="pca")
    print(f"Task {_task}: {cell_embs_pacmap[_task].shape}")

#### Now we can plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%config InlineBackend.figure_format = 'retina'
%config InlineBackend.figure_dpi = 300

In [ ]:
try:
    plt.clf()
except:
    pass

# A4纸的宽度约为21厘米（8.27英寸），高度可以根据需要调整
# a4_width_cm = 21
# cm_to_inches = 0.393701
# a4_width_inches = a4_width_cm * cm_to_inches
n_cols = 2
n_rows = len(cell_embs_pacmap)

fig_width = 2.6 * n_cols  # 调整图表的整体宽度以适应列数
fig_height = 2.4 * n_rows  # 调整图表的整体高度以适应行数

sns.set_theme(style="ticks")
sns.set_context("paper", font_scale=1.0)  # 设置图表的上下文和字体大小

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(fig_width, fig_height),
    sharex=False,  # 不共享 x 轴
    sharey=False,  # 不共享 y 轴
)
if n_rows == 1:
    axes = axes.reshape(1, -1)

# sns.scatterplot(x=X_transformed[:, 0], y=X_transformed[:, 1], hue=y_true_text, palette="tab10", s=4, ax=ax[0])
# sns.scatterplot(x=X_transformed[:, 0], y=X_transformed[:, 1], hue=y_pred_text, palette="tab10", s=4, ax=ax[1])

for i, _task in enumerate(cell_embs_pacmap.keys()):
    ax = axes[i, 0]
    ax.set_title(_tasks_text[i])
    sns.scatterplot(x=cell_embs_pacmap[_task][:, 0], y=cell_embs_pacmap[_task][:, 1], hue=y_true_text, palette="colorblind", s=4, ax=ax, legend=False)
    # 删除框线
    for spine in ax.spines.values():
        spine.set_visible(False)
    # 删除刻度棒和刻度数字
    ax.tick_params(axis='both', which='both', length=0)  # 隐藏刻度棒
    ax.set_xticks([])  # 清空 x 轴刻度
    ax.set_yticks([])  # 清空 y 轴刻度

    ax = axes[i, 1]
    ax.set_title(_tasks_text[i])
    sns.scatterplot(x=cell_embs_pacmap[_task][:, 0], y=cell_embs_pacmap[_task][:, 1], hue=ys_pred_text[_task], palette="colorblind", s=4, ax=ax, legend=False)
    # 删除框线
    for spine in ax.spines.values():
        spine.set_visible(False)
    # 删除刻度棒和刻度数字
    ax.tick_params(axis='both', which='both', length=0)  # 隐藏刻度棒
    ax.set_xticks([])  # 清空 x 轴刻度
    ax.set_yticks([])  # 清空 y 轴刻度

fig.tight_layout(pad=0.1)
fig.show()

# Save as PNG and PDF
fig.savefig(f".fig/{fig_name}.png", dpi=300)
fig.savefig(f".fig/{fig_name}.pdf")